
# ⚡ Energy Forecasting with XGBoost: Line-by-Line Explanation

This notebook provides a **comprehensive explanation** of how to build, evaluate, and visualize an XGBoost regression model using household power consumption features.

---

### 📦 Installing and Importing Required Libraries

```python
!pip install xgboost --quiet
```
* Installs the `xgboost` library if it is not already available in the environment.

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
```
* Imports libraries for:
  - **`pandas`**: loading and handling tabular data
  - **`numpy`**: numerical computations
  - **`matplotlib.pyplot`**: plotting and visualization
  - **`xgboost`**: gradient boosting algorithm for supervised learning

---

### 📥 Load Dataset

```python
df = pd.read_csv('data/features.csv')
```
* Reads the dataset containing features and target values for energy forecasting.

```python
X = df.drop(columns=['Global_active_power', 'datetime'])
y = df['Global_active_power']
```
* `X`: contains input features (excluding the target and timestamp).
* `y`: the target variable, `Global_active_power`, which we want to predict.

---

### 🧱 Create DMatrix for XGBoost

```python
dtrain = xgb.DMatrix(X, label=y)
```
* Converts the dataset into XGBoost’s optimized internal data structure `DMatrix`, which is faster and more efficient for training.

---

### ⚙️ Set XGBoost Parameters

```python
params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}
```

| Parameter | Description |
|-----------|-------------|
| `objective` | Regression task using squared error loss |
| `max_depth` | Maximum tree depth (controls model complexity) |
| `eta` | Learning rate (step size shrinkage) |
| `subsample` | Proportion of training samples used per boosting round |
| `colsample_bytree` | Fraction of features used per tree |
| `seed` | Random seed for reproducibility |

---

### 🔁 Cross-Validation

```python
cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=100,
    nfold=5,
    metrics='rmse',
    early_stopping_rounds=10,
    verbose_eval=False
)
```
* Performs 5-fold cross-validation with up to 100 boosting rounds.
* Uses **RMSE** as the evaluation metric.
* Stops early if performance doesn’t improve over 10 rounds.

---

```python
cv_results.tail()
```
* Shows the last few entries of cross-validation results, including training and test RMSE per round.

---

### 🏋️ Train Final Model

```python
model = xgb.train(params=params, dtrain=dtrain, num_boost_round=cv_results.shape[0])
```
* Trains the final model using the optimal number of boosting rounds based on CV results.

---

### 💾 Save the Model

```python
model.save_model('models/xgb_energy_model.json')
```
* Saves the trained XGBoost model in JSON format for future reuse or deployment.

---

### 📈 Visualize Learning Curve

```python
plt.figure(figsize=(10, 5))
plt.plot(cv_results['train-rmse-mean'], label='Train')
plt.plot(cv_results['test-rmse-mean'], label='Test')
plt.title('Learning Curve (RMSE)')
plt.xlabel('Boosting Round')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True)
plt.show()
```
* Plots RMSE over boosting rounds for both training and validation sets.
* Helps identify underfitting or overfitting based on the divergence of curves.

---

### 🧠 Feature Importance Plot

```python
xgb.plot_importance(model, max_num_features=10)
plt.title('Top 10 Feature Importances')
plt.show()
```
* Displays the top 10 most important features based on their contribution to prediction.
* Helps interpret model decisions and prioritize influential variables.

---

### ✅ Summary

This notebook demonstrates:
1. Loading and preparing data.
2. Initializing and tuning an XGBoost regressor.
3. Evaluating via cross-validation.
4. Visualizing model performance and feature importance.
